In [4]:
from river.datasets import synth
from river import metrics
from river.drift import ADWIN
from river.ensemble import SRPClassifier as VanillaSRPClassifier # Import original as alias
from src.streaming_random_patches import SRPClassifier

%load_ext autoreload
%autoreload 2

In [14]:
print("--- 🏎️ STARTING THE RACE: Vanilla SRP vs. Custom C-DES ---")

# 1. Set up the exact same violent drift stream we used before
stream = synth.ConceptDriftStream(
    stream=synth.RandomTree(seed_tree=1, seed_sample=42),
    drift_stream=synth.RandomTree(seed_tree=99, seed_sample=42),
    position=2000,
    width=50,
    seed=123
).take(4000)

# 2. Initialize Vanilla River SRP (The Baseline)
vanilla_model = VanillaSRPClassifier(
    n_models=10,
    drift_detector=ADWIN(),
    warning_detector=None, # Disabled to match
    seed=42
)

# 3. Initialize Your Custom C-DES SRP (The Challenger)
cdes_model = SRPClassifier(
    n_models=10, 
    n_clusters=3,               
    drift_detector=ADWIN(),
    warning_detector=None, 
    training_method="patches",
    seed=42
)

# 4. Set up independent score trackers
vanilla_metric = metrics.Accuracy() + metrics.MacroF1()
cdes_metric = metrics.Accuracy() + metrics.MacroF1()

vanilla_drifts = 0
cdes_drifts = 0

--- 🏎️ STARTING THE RACE: Vanilla SRP vs. Custom C-DES ---


In [15]:
# 5. The Race Loop
for i, (x, y) in enumerate(stream):
    # --- Predict Phase ---
    y_pred_vanilla = vanilla_model.predict_one(x)
    y_pred_cdes = cdes_model.predict_one(x)
    
    # --- Update Metrics ---
    if y_pred_vanilla is not None:
        vanilla_metric.update(y, y_pred_vanilla)
    if y_pred_cdes is not None:
        cdes_metric.update(y, y_pred_cdes)
        
    # --- Train Phase ---
    vanilla_model.learn_one(x, y)
    cdes_model.learn_one(x, y)
    
    # --- Track Drifts (using the first tree as a proxy) ---
    if vanilla_model.models[0].drift_detector.drift_detected:
        vanilla_drifts += 1
    if cdes_model.models[0].drift_detector.drift_detected:
        cdes_drifts += 1

    # --- Print Lap Times ---
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} \n\n Vanilla: \n {vanilla_metric} \n\n C-DES \n {cdes_metric} \n")

Instance 1000 

 Vanilla: 
 Accuracy: 64.46%
MacroF1: 60.89% 

 C-DES 
 Accuracy: 63.86%
MacroF1: 60.23% 

Instance 2000 

 Vanilla: 
 Accuracy: 68.83%
MacroF1: 65.45% 

 C-DES 
 Accuracy: 68.53%
MacroF1: 65.11% 

Instance 3000 

 Vanilla: 
 Accuracy: 64.92%
MacroF1: 62.55% 

 C-DES 
 Accuracy: 63.85%
MacroF1: 61.22% 

Instance 4000 

 Vanilla: 
 Accuracy: 63.42%
MacroF1: 62.07% 

 C-DES 
 Accuracy: 62.62%
MacroF1: 60.92% 



In [16]:
print("\n" + "="*40)
print("🏆 FINAL RACE RESULTS 🏆")
print("="*40)
print(f"VANILLA SRP:")
print(f"  {vanilla_metric}")
print(f"  Drifts triggered in Model 0: {vanilla_drifts}")
print("-" * 40)
print(f"CUSTOM C-DES SRP:")
print(f"  {cdes_metric}")
print(f"  Drifts triggered in Model 0: {cdes_drifts}")


🏆 FINAL RACE RESULTS 🏆
VANILLA SRP:
  Accuracy: 63.42%
MacroF1: 62.07%
  Drifts triggered in Model 0: 0
----------------------------------------
CUSTOM C-DES SRP:
  Accuracy: 62.62%
MacroF1: 60.92%
  Drifts triggered in Model 0: 1


In [17]:
print("--- 🏎️ STARTING RACE 2: The AGRAWAL Localized Drift Test ---")

# 1. Set up the AGRAWAL Stream (Simulating localized demographic drift)
stream = synth.ConceptDriftStream(
    stream=synth.Agrawal(classification_function=0, seed=42),
    drift_stream=synth.Agrawal(classification_function=2, seed=42),
    position=2000,
    width=50,
    seed=123
).take(4000)

# 2. Initialize Vanilla River SRP
vanilla_model = VanillaSRPClassifier(
    n_models=10,
    drift_detector=ADWIN(),
    warning_detector=None,
    seed=42
)

# 3. Initialize Your Custom C-DES SRP
cdes_model = SRPClassifier(
    n_models=10, 
    n_clusters=3,               
    drift_detector=ADWIN(),
    warning_detector=None, 
    training_method="patches",
    seed=42
)

# 4. Set up independent score trackers
vanilla_metric = metrics.Accuracy() + metrics.MacroF1()
cdes_metric = metrics.Accuracy() + metrics.MacroF1()

vanilla_drifts = 0
cdes_drifts = 0

--- 🏎️ STARTING RACE 2: The AGRAWAL Localized Drift Test ---


In [18]:
# 5. The Race Loop
for i, (x, y) in enumerate(stream):
    # Predict Phase
    y_pred_vanilla = vanilla_model.predict_one(x)
    y_pred_cdes = cdes_model.predict_one(x)
    
    # Update Metrics
    if y_pred_vanilla is not None:
        vanilla_metric.update(y, y_pred_vanilla)
    if y_pred_cdes is not None:
        cdes_metric.update(y, y_pred_cdes)
        
    # Train Phase
    vanilla_model.learn_one(x, y)
    cdes_model.learn_one(x, y)
    
    # Track Drifts (Model 0 proxy)
    if vanilla_model.models[0].drift_detector.drift_detected:
        vanilla_drifts += 1
    if cdes_model.models[0].drift_detector.drift_detected:
        cdes_drifts += 1

    # Print Lap Times
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} \n\n Vanilla: \n {vanilla_metric} \n\n C-DES \n {cdes_metric} \n")

Instance 1000 

 Vanilla: 
 Accuracy: 97.10%
MacroF1: 96.75% 

 C-DES 
 Accuracy: 97.00%
MacroF1: 96.65% 

Instance 2000 

 Vanilla: 
 Accuracy: 98.00%
MacroF1: 97.73% 

 C-DES 
 Accuracy: 98.05%
MacroF1: 97.79% 

Instance 3000 

 Vanilla: 
 Accuracy: 85.16%
MacroF1: 83.56% 

 C-DES 
 Accuracy: 93.70%
MacroF1: 93.27% 

Instance 4000 

 Vanilla: 
 Accuracy: 79.47%
MacroF1: 77.56% 

 C-DES 
 Accuracy: 94.55%
MacroF1: 94.28% 



In [19]:
print("\n" + "="*40)
print("🏆 FINAL AGRAWAL RACE RESULTS 🏆")
print("="*40)
print(f"VANILLA SRP:")
print(f"  {vanilla_metric}")
print(f"  Drifts triggered in Model 0: {vanilla_drifts}")
print("-" * 40)
print(f"CUSTOM C-DES SRP:")
print(f"  {cdes_metric}")
print(f"  Drifts triggered in Model 0: {cdes_drifts}")


🏆 FINAL AGRAWAL RACE RESULTS 🏆
VANILLA SRP:
  Accuracy: 79.47%
MacroF1: 77.56%
  Drifts triggered in Model 0: 0
----------------------------------------
CUSTOM C-DES SRP:
  Accuracy: 94.55%
MacroF1: 94.28%
  Drifts triggered in Model 0: 2
